# FF-STEM fusion demo: Au nanoparticle, low dose

This notebook reconstructs a real 4D-STEM acquisition with `scatterem`'s
FF-STEM fusion pipeline -- direct (SSB) ptychography, tilt-corrected dark
field, and their SSNR-weighted fusion -- end to end on a free Colab T4 GPU.

**Paper:** S. You, G. Varnavides, S. Khavnekar, N. Palatkin, S. Shao, M. Wu,
D. Stroppa, D. Chernikova, B. Zhu, R. Egoavil, S. Vespucci, D. Krishnan,
X. Ye,
F. K. M. Schur, E. Spiecker, P. Pelz, *"Gap-Free Information Transfer in
4D-STEM via Fusion of Complementary Scattering Channels"*, **Advanced
Science** (2026), doi:[10.1002/advs.76620](https://doi.org/10.1002/advs.76620).

**Data:** Zenodo record [18008901](https://zenodo.org/records/18008901)
(DOI 10.5281/zenodo.18008901), licensed **CC-BY-4.0**. If you use this data,
please cite the record and the paper above.

**The three channels:**

- **Direct (SSB) ptychography** reconstructs gap-free low/mid spatial
  frequencies from the phase gradient encoded in the bright-field disk alone.
- **Tilt-corrected dark field (tcDF)** recovers the complementary higher
  spatial frequencies from the parallax shift of dark-field scattering
  (outside the bright-field disk).
- **Fused full field (FFF)** combines the two as an SSNR-weighted Wiener
  (Fourier-domain) fusion, giving a single gap-free image with no missing
  frequency band.

## Before you run this

Only one thing: **a GPU runtime**. *Runtime > Change runtime type > T4 GPU*.

Both the code and the Zenodo data are public, so no tokens, accounts or
credentials are involved. The install cell below fetches `scatterem` straight
from GitHub, and the dataset downloads itself on first use, verified by md5.

This notebook reconstructs the Au low-dose acquisition, and it sizes itself to
the GPU it finds. On a free T4 it uses a reduced scan (128x128 -> 512x512
output, about 4.4 GiB); given 20 GiB or more it runs the paper's own
configuration (256x256 -> 1024x1024, about 17.6 GiB). The cell below reports
which it chose, and you can override `SCAN_EDGE_CROP` there.


## 1. Check the runtime

In [ ]:
import subprocess

try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=True,
    )
except (FileNotFoundError, subprocess.CalledProcessError):
    raise SystemExit(
        "No NVIDIA GPU detected. In Colab: Runtime > Change runtime type > "
        "Hardware accelerator > T4 GPU, then Runtime > Restart session and "
        "re-run this cell."
    )

gpu_name, mem_str = (s.strip() for s in smi.stdout.strip().split(","))
TOTAL_GIB = float(mem_str.split()[0]) / 1024.0  # MiB -> GiB
print(f"GPU: {gpu_name}  ({TOTAL_GIB:.1f} GiB total)")


## 2. Install `scatterem`

The GitHub token is read from Colab Secrets (never typed or pasted into a
cell) and handed to `pip` through the shell's own environment-variable
expansion, so it never appears in this notebook's source or its output.


In [ ]:
# Install scatterem from the public repository.
#
# Both this code and the Zenodo data are public, so there is nothing to
# authenticate. Warp compiles a few CUDA kernels on first import, which takes a
# moment; that is expected.
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "git+https://github.com/scatterem/scatterem",
    ],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stdout[-4000:])
    print(result.stderr[-4000:])
    raise SystemExit(
        "scatterem failed to install; the output above should say why."
    )
print("scatterem installed")


## 3. Choose a configuration that fits this GPU

The paper's own configuration (`scan_edge_crop=128`, a 256x256 scan
upsampled to a 1024x1024 output) peaks at **17.6 GiB** of GPU memory, which
does not fit a free Colab T4 (~13 GiB usable). Measured peak memory for the
Au low-dose dataset at a few crop settings:

| `scan_edge_crop` | scan | output | peak GPU memory | fits free T4? |
| --- | --- | --- | --- | --- |
| 128 (paper's) | 256x256 | 1024x1024 | 17.6 GiB | no |
| 192 | 128x128 | 512x512 | 4.4 GiB | yes |
| 224 | 64x64 | 256x256 | 2.5 GiB | yes |

Note: raising `n_batches` *increases* peak memory (15/60/150 batches gave
17.6/21.0/24.5 GiB, bit-identical output), and halving `upsample` made things
slightly worse (18.3 GiB). Shrinking the scan via `scan_edge_crop` is the
only effective lever for fitting a smaller GPU -- this notebook does not
touch `n_batches` or `upsample` to "optimize" memory.


In [ ]:
if TOTAL_GIB >= 20:
    SCAN_EDGE_CROP = 128
    PROFILE = "paper (256x256 scan -> 1024x1024 output, ~17.6 GiB peak)"
else:
    SCAN_EDGE_CROP = 192
    PROFILE = "reduced (128x128 scan -> 512x512 output, ~4.4 GiB peak)"

print(f"Selected profile: {PROFILE}")
print(f"scan_edge_crop = {SCAN_EDGE_CROP}")


## 4. Load the dataset

`You2026AuLowDose` downloads the three Zenodo files it needs (the Dectris
master file plus its two external data blocks, ~500 MB total) and verifies
each against its published md5 checksum. Constructing it also applies the
acquisition repairs that are properties of this specific dataset rather than
a reconstruction choice: reshaping the raw frame stream into a square scan
grid, replacing a known-bad 2x2 detector patch with the mean of its
surrounding ring, and cropping `scan_edge_crop` unreliable rows/columns from
each scan edge. Finally it calibrates the reciprocal-space pixel size `dk`
from the measured bright-field disk radius.


In [ ]:
from scatterem.datasets import You2026AuLowDose

dataset = You2026AuLowDose(
    root="data",
    download=True,
    device="cuda",
    scan_edge_crop=SCAN_EDGE_CROP,
)
print(dataset)
print(f"scan shape: {tuple(int(s) for s in dataset.array.shape[:2])}")
print(f"detector shape: {tuple(int(s) for s in dataset.detector_shape)}")
print(f"dk = {dataset.dk} 1/Angstrom")
print(f"fluence = {dataset.fluence:.3g} e-/Angstrom^2")


## 5. Fit the aberrations

Parameters below are the paper's Figure 4 script's: a sharpness-based
("autofocus") fit of aberrations up to 2nd order, maximizing total variation
(`sharpness_metric="tv"`) over a 150x150 region-of-interest, seeded from an
initial defocus guess.


In [ ]:
dataset.meta.aberrations.array[0] = -150.0
dataset.determine_aberrations_(
    bright_field_mask_threshold=0.1,
    correction_method="autofocus",
    sharpness_metric="tv",
    bin_factors=(1, 1, 1),
    verbosity=1,
    correct_order=2,
    num_iterations=50,
    lr=1,
    roi_shape=(150, 150),
    upsample=1.0,
)


## 6. Reconstruct the three channels

In [ ]:
import torch

direct_ptycho_image, ssnr_ptycho = dataset.direct_ptychography(
    upsample="nyquist",
    n_batches=25,
    return_snr=True,
    verbosity=1,
)

tcDF, ssnr_tcdf = dataset.tilt_corrected_dark_field(
    n_dark_field_segments=16,
    verbosity=0,
    bright_field_mask_threshold=0.1,
    upsample="nyquist",
    return_snr=True,
)

# verbosity=1 (not Fig4_diffraction.py's verbosity=2) -- deliberately quieter
# for a Colab cell; verbosity=2 also plots the direct-ptychography/tcDF/FFF
# intermediates that this notebook already shows in the next section.
fff, phase_weighted, tcdf_weighted = dataset.fused_full_field(
    verbosity=1,
    bright_field_mask_threshold=0.3,
)

print(f"direct ptychography: {tuple(direct_ptycho_image.shape)}")
print(f"tilt-corrected dark field: {tuple(tcDF.shape)}")
print(f"fused full field: {tuple(fff.shape)}")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")


## 7. Look at the results

The scale bar below is physical, in Angstrom. The reconstructions are
upsampled relative to the scan, so one image pixel is the scan step scaled
by (scan pixels / image pixels) -- not the raw scan step itself.


In [ ]:
import matplotlib.pyplot as plt

import scatterem.vis as vis
from scatterem.utils.data import Sampling


def image_sampling(img):
    ny, nx = img.shape[-2:]
    return Sampling(
        pixel_size=(
            float(dataset.sampling[0]) * int(dataset.array.shape[0]) / ny,
            float(dataset.sampling[1]) * int(dataset.array.shape[1]) / nx,
        ),
        units=("Å", "Å"),
    )


for img, title in (
    (direct_ptycho_image, "Direct ptychography (phase)"),
    (tcDF, "Tilt-corrected dark field"),
    (fff, "Fused full field"),
):
    fig, ax = plt.subplots(figsize=(6, 6))
    vis.show_2d(
        [img],
        cbar=True,
        title=[title],
        figax=(fig, ax),
        sampling=image_sampling(img),
        clip_percentile=(2, 98),
    )
    plt.show()


## Other datasets

The same Zenodo record (18008901) also has three other named datasets in
`scatterem.datasets`, each carrying its own acquisition constants:

- `You2026Carbon` -- amorphous carbon (paper Figure 2). The lightest of the
  four: ~1.2 GiB peak GPU memory.
- `You2026Co3O4` -- Co3O4 nanoparticles (paper Figure 2).
- `You2026Gd2O3` -- Gd2O3 nanoparticles, the paper's Figure 1. This one is
  much larger: a 6.6 GB download and ~19.6 GiB peak GPU memory, so it will
  **not** run on a free Colab runtime.

For the full set of figure-reproduction scripts (including comparison plots
against independent parallax reconstructions), see
`experiments/ff_stem/` in the repository.
